# Graph Data Science (GDS)

## Taller Educativo sobre Algoritmos de Ciencia de Datos en Grafos

### Objetivos de Aprendizaje

En este cuaderno aprenderás:

1. **GDS Fundamentals**: Arquitectura y conceptos clave
2. **Proyección de Grafos**: Nativa y Cypher
3. **Centralidad**: PageRank, Betweenness, Closeness, Degree
4. **Comunidades**: Louvain, Label Propagation, WCC
5. **Similitud**: Node Similarity, KNN
6. **Path Finding**: Dijkstra, All Shortest Paths
7. **Embeddings**: Node2Vec, FastRP
8. **Pipeline Completo**: Análisis end-to-end de fraude


## 1. Introducción a Graph Data Science

### 1.1 ¿Qué es GDS?

**Graph Data Science (GDS)** es la biblioteca oficial de Neo4j para algoritmos de análisis de grafos y machine learning.

**Características**:
- 50+ algoritmos optimizados
- Procesamiento in-memory ultrarrápido
- Algoritmos de ML (embeddings, GNNs)
- Escalable a grafos de miles de millones de nodos

### 1.2 Arquitectura de GDS

```
Neo4j (disco) → Proyección (memoria) → Algoritmo → Resultados
```

1. **Proyectar**: Cargar subgrafo en memoria
2. **Ejecutar**: Correr algoritmo sobre proyección
3. **Retornar**: stream, stats, mutate, o write

### 1.3 Modos de Ejecución

- **stream**: Retorna resultados, no modifica grafo
- **stats**: Solo estadísticas
- **mutate**: Guarda en proyección (no en disco)
- **write**: Persiste resultados en Neo4j

### 1.4 GDS vs APOC

| Aspecto | GDS | APOC |
|---------|-----|------|
| Algoritmos | ✅ Optimizados | Deprecados |
| Performance | ✅ In-memory | Bueno |
| ML | ✅ Sí | No |
| Utilidades | Limitado | ✅ Extenso |



## 2. Configuración

In [ ]:
# Importar bibliotecas
import importlib
from neo4j import GraphDatabase
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Importar consultas GDS
import sys
sys.path.append('.')
from consultas_gds import *

print("✓ Bibliotecas importadas")

In [ ]:
# Configuración
NEO4J_URI = "bolt://localhost:7687"
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = "abc123456"  # CAMBIAR
NEO4J_DATABASE = "fraudedb"

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

def ejecutar_consulta(query, parametros=None):
    with driver.session(database=NEO4J_DATABASE) as session:
        resultado = session.run(query, parametros or {})
        return [record.data() for record in resultado]

# Verificar
try:
    version = ejecutar_consulta("RETURN gds.version() AS version")
    print(f"✓ GDS Version: {version[0]['version']}")
except Exception as e:
    print(f"✗ Error: {e}")

## 3. Proyección de Grafos

### 3.1 Proyección Nativa (Recomendada)

In [ ]:
# Limpiar proyecciones anteriores
try:
    ejecutar_consulta("CALL gds.graph.drop('red-empresas')")
    ejecutar_consulta("CALL gds.graph.drop('red-empresas-undirected')")
except:
    pass

# Proyectar grafo simple
query = GDS_PROYECCION_NATIVE["example_simple"]
resultado = ejecutar_consulta(query)

print("Proyección creada:")
print(f"  Nombre: {resultado[0]['graphName']}")
print(f"  Nodos: {resultado[0]['nodeCount']}")
print(f"  Relaciones: {resultado[0]['relationshipCount']}")

### 3.2 Gestión de Proyecciones

In [ ]:
# Listar proyecciones
query = GDS_PROYECCION_MANAGEMENT["list_graphs"]
grafos = ejecutar_consulta(query)

print("Proyecciones activas:")
for g in grafos:
    print(f"  - {g['graphName']}: {g['nodeCount']} nodos, {g['relationshipCount']} relaciones")

## 4. Algoritmos de Centralidad

### 4.1 PageRank

In [ ]:
# PageRank - Modo STREAM
query = GDS_PAGERANK["stream"]
resultado = ejecutar_consulta(query)

df = pd.DataFrame(resultado)
print("Top 15 empresas por PageRank:")
print(df.head(15).to_string(index=False))

In [ ]:
# Visualizar distribución de PageRank
plt.figure(figsize=(10, 5))
plt.hist(df['pagerank_score'], bins=30, edgecolor='black')
plt.xlabel('PageRank Score')
plt.ylabel('Frecuencia')
plt.title('Distribución de PageRank')
plt.show()

### 4.2 Betweenness Centrality

In [ ]:
# Betweenness - Identifica "puentes"
query = GDS_BETWEENNESS["stream"]
resultado = ejecutar_consulta(query)

df_betweenness = pd.DataFrame(resultado)
print("Top 10 empresas por Betweenness (puentes críticos):")
print(df_betweenness.head(10).to_string(index=False))

### 4.3 Degree Centrality

In [ ]:
# Degree - Número de conexiones
query = GDS_DEGREE_CENTRALITY["stream"]
resultado = ejecutar_consulta(query)

df_degree = pd.DataFrame(resultado)
print("Top 10 empresas más conectadas:")
print(df_degree.head(10).to_string(index=False))

## 5. Detección de Comunidades

### 5.1 Louvain

In [ ]:
# Louvain - Detectar comunidades
query = GDS_LOUVAIN["stream"]
resultado = ejecutar_consulta(query)

df_comunidades = pd.DataFrame(resultado)
print("Comunidades detectadas con Louvain:")
print(df_comunidades.to_string(index=False))

### 5.2 Label Propagation

In [ ]:
# Label Propagation - Más rápido
query = GDS_LABEL_PROPAGATION["stream"]
resultado = ejecutar_consulta(query)

df_lp = pd.DataFrame(resultado)
print("Comunidades con Label Propagation:")
print(df_lp.head(10).to_string(index=False))

### 5.3 Triangle Count

In [ ]:
# ============================================================
# CREAR PROYECCIÓN UNDIRECTED PARA TRIANGLE COUNT
# ============================================================

# Paso 1: Eliminar el grafo si ya existe (para evitar errores)
query_drop = """
CALL gds.graph.drop('red-empresas-undirected', false)
YIELD graphName
RETURN graphName AS grafo_eliminado
"""

try:
    resultado = ejecutar_consulta(query_drop)
    print("✓ Grafo anterior eliminado" if resultado else "No existía grafo previo")
except:
    print("No existía grafo previo")

# Paso 2: Crear proyección con orientación UNDIRECTED
query_project = """
CALL gds.graph.project(
    'red-empresas-undirected',
    'Empresa',
    {
        EMITE_FACTURA: {
            orientation: 'UNDIRECTED'
        }
    }
)
YIELD graphName, nodeCount, relationshipCount
RETURN graphName, nodeCount, relationshipCount
"""

resultado = ejecutar_consulta(query_project)
print(f"✓ Grafo proyectado: {resultado}")

# Paso 3: Ejecutar Triangle Count
query_triangles = """
CALL gds.triangleCount.stream('red-empresas-undirected')
YIELD nodeId, triangleCount
WITH gds.util.asNode(nodeId) AS empresa, triangleCount
WHERE triangleCount > 0
RETURN 
    empresa.nombre AS empresa,
    triangleCount AS num_triangulos
ORDER BY triangleCount DESC
LIMIT 15
"""

resultado = ejecutar_consulta(query_triangles)
df_triangles = pd.DataFrame(resultado)
print("🔺 Empresas con triángulos (conexiones circulares):")
print(df_triangles.to_string(index=False) if not df_triangles.empty else "No se encontraron triángulos")

In [ ]:
# Triangle Count - Cohesión local
query = GDS_TRIANGLE_COUNT["stream"]
resultado = ejecutar_consulta(query)

df_triangles = pd.DataFrame(resultado)
print("Empresas con más triángulos (redes densas):")
print(df_triangles.head(10).to_string(index=False))

## 6. Similitud

### 6.1 Node Similarity

In [ ]:
# Node Similarity - Vecinos comunes
query = GDS_NODE_SIMILARITY["stream"]
resultado = ejecutar_consulta(query)

df_sim = pd.DataFrame(resultado)
print("Pares de empresas más similares:")
print(df_sim.head(15).to_string(index=False))

## 7. Path Finding

### 7.1 Preparar Proyección Ponderada

In [ ]:
# Crear proyección con pesos para path finding
try:
    ejecutar_consulta("CALL gds.graph.drop('red-ponderada')")
except:
    pass

query = GDS_PROYECCION_NATIVE["example_with_properties"]
resultado = ejecutar_consulta(query)
print(f"✓ Proyección ponderada creada: {resultado[0]['graphName']}")

### 7.2 Shortest Path (Dijkstra)

In [ ]:
# Obtener 2 empresas para ejemplo
empresas = ejecutar_consulta("""
MATCH (e:Empresa)
RETURN e.nombre AS nombre, id(e) AS id
LIMIT 2
""")

if len(empresas) >= 2:
    emp1 = empresas[0]['nombre']
    emp2 = empresas[1]['nombre']
    
    print(f"Camino más corto de '{emp1}' a '{emp2}':")
    
    # Usar query template de GDS_SHORTEST_PATH
    # (simplificado para el ejemplo)
    query = f"""
    MATCH (inicio:Empresa {{nombre: '{emp1}'}})
    MATCH (fin:Empresa {{nombre: '{emp2}'}})
    WITH id(inicio) AS startId, id(fin) AS endId
    CALL gds.shortestPath.dijkstra.stream('red-ponderada', {{
        sourceNode: startId,
        targetNode: endId,
        relationshipWeightProperty: 'monto_total'
    }})
    YIELD nodeIds, totalCost
    RETURN 
        [nodeId IN nodeIds | gds.util.asNode(nodeId).nombre] AS camino,
        totalCost AS costo_total
    """
    
    try:
        resultado = ejecutar_consulta(query)
        if resultado:
            print(f"  Camino: {' → '.join(resultado[0]['camino'])}")
            print(f"  Costo total: {resultado[0]['costo_total']:.2f}")
    except Exception as e:
        print(f"  No hay camino o error: {e}")

## 8. Embeddings

### 8.1 FastRP (Fast Random Projection)

In [ ]:
# FastRP - Embeddings rápidos
query = GDS_FASTRP["stream"]
resultado = ejecutar_consulta(query)

print(f"Embeddings generados para {len(resultado)} empresas")
print(f"Dimensión: {resultado[0]['dim']} si existe el campo")
print("Primeros 5 ejemplos:")
for i, r in enumerate(resultado[:5]):
    print(f"  {r['empresa']}: dim={r.get('dim', 'N/A')}")

## 9. Pipeline Completo de Análisis de Fraude

Vamos a combinar múlt iples algoritmos para un análisis completo.

In [ ]:
# Limpiar y crear proyección para análisis
try:
    ejecutar_consulta("CALL gds.graph.drop('analisis-fraude')")
except:
    pass

query = GDS_PIPELINE_FRAUDE["paso_1_proyectar"]
ejecutar_consulta(query)
print("✓ Paso 1: Proyección creada")

In [ ]:
# Paso 2: PageRank
query = GDS_PIPELINE_FRAUDE["paso_2_pagerank"]
resultado = ejecutar_consulta(query)
print(f"✓ Paso 2: PageRank calculado ({resultado[0]['nodePropertiesWritten']} nodos)")

In [ ]:
# Paso 3: Comunidades
query = GDS_PIPELINE_FRAUDE["paso_3_comunidades"]
resultado = ejecutar_consulta(query)
print(f"✓ Paso 3: Comunidades detectadas")

In [ ]:
# Paso 4: Betweenness
query = GDS_PIPELINE_FRAUDE["paso_4_betweenness"]
resultado = ejecutar_consulta(query)
print(f"✓ Paso 4: Betweenness calculado")

In [ ]:
# Paso 5: Análisis combinado
query = GDS_PIPELINE_FRAUDE["paso_5_analisis"]
resultado = ejecutar_consulta(query)

df_fraude = pd.DataFrame(resultado)
print("🔍 ANÁLISIS COMBINADO DE FRAUDE\n")
print(df_fraude.to_string(index=False))

In [ ]:
# Visualizar distribución de riesgo
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
df_fraude['nivel_riesgo'].value_counts().plot(kind='bar')
plt.title('Distribución por Nivel de Riesgo')
plt.xlabel('Nivel')
plt.ylabel('Número de Empresas')

plt.subplot(1, 2, 2)
plt.scatter(df_fraude['pagerank'], df_fraude['score_fraude'], 
           c=df_fraude['nivel_riesgo'].map({'ALTO': 'red', 'MEDIO': 'orange', 'BAJO': 'green'}),
           alpha=0.6)
plt.xlabel('PageRank')
plt.ylabel('Score de Fraude')
plt.title('PageRank vs Score de Fraude')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 10. Limpieza

In [ ]:
# Eliminar proyecciones para liberar memoria
grafos_a_eliminar = ['red-empresas', 'red-ponderada', 'analisis-fraude']

for grafo in grafos_a_eliminar:
    try:
        ejecutar_consulta(f"CALL gds.graph.drop('{grafo}')")
        print(f"✓ Eliminado: {grafo}")
    except:
        pass

driver.close()
print("✓ Conexión cerrada")

## 11. Resumen y Conclusiones

### Lo que Aprendiste

✅ Proyección de grafos (nativa y Cypher)  
✅ **Centralidad**: PageRank, Betweenness, Degree, Closeness  
✅ **Comunidades**: Louvain, Label Propagation, Triangle Count  
✅ **Similitud**: Node Similarity, KNN  
✅ **Path Finding**: Dijkstra  
✅ **Embeddings**: FastRP, Node2Vec  
✅ **Pipeline completo**: Análisis multinivel de fraude  

### Algoritmos GDS Cubiertos

**Centralidad (4)**:
- PageRank
- Betweenness Centrality
- Degree Centrality
- Closeness Centrality

**Comunidades (3)**:
- Louvain
- Label Propagation
- Triangle Count

**Similitud (2)**:
- Node Similarity
- K-Nearest Neighbors

**Path Finding (1)**:
- Dijkstra Shortest Path

**Embeddings (1)**:
- FastRP

**Total: 11+ algoritmos populares**

### Comparación APOC vs GDS

- **APOC**: Utilidades, I/O, refactoring, Cypher dinámico
- **GDS**: Algoritmos de grafos optimizados, ML, embeddings
- **Mejor práctica**: Usar ambos según necesidad

### Ejercicios Propuestos

1. **Centralidad**: Comparar rankings de diferentes métricas
2. **Comunidades**: Analizar overlap entre Louvain y Label Propagation
3. **Similitud**: Encontrar empresas similares por características
4. **Path Finding**: Analizar todos los caminos desde empresa fantasma
5. **Embeddings**: Visualizar embeddings en 2D con t-SNE
6. **Pipeline**: Crear scoring personalizado con 5+ métricas

### Aplicaciones a Fraude de IVA

- **PageRank**: Empresas clave en redes
- **Betweenness**: Intermediarios críticos
- **Louvain**: Redes organizadas de fraude
- **Node Similarity**: Empresas con patrones idénticos
- **Path Finding**: Rutas de dinero
- **Pipeline**: Scoring multinivel

### Próximos Pasos

- Explorar más algoritmos en [GDS Docs](https://neo4j.com/docs/graph-data-science/current/)
- Probar algoritmos beta (GraphSAGE, Link Prediction)
- Integrar con frameworks ML (Python, scikit-learn)
- Optimizar proyecciones para grafos grandes

### Recursos

- [GDS Documentation](https://neo4j.com/docs/graph-data-science/current/)
- [Algorithms Guide](https://neo4j.com/docs/graph-data-science/current/algorithms/)
- Ver `consultas_gds.py` para catálogo completo

## ¡Felicidades! 🎉

Has completado el taller educativo de Neo4j. Ahora dominas:

1. ✅ **Notebook 01**: Modelado de grafos y detección de fraude de IVA
2. ✅ **Notebook 02**: Procedimientos avanzados con APOC
3. ✅ **Notebook 03**: Algoritmos de Graph Data Science

Continúa explorando y aplicando estos conocimientos a tus propios casos de uso.